# HR Analytics – Dự đoán nghỉ việc (Attrition)
Notebook này xây dựng pipeline mô hình hóa với các cải tiến: đánh giá bằng **Stratified CV**, chọn metric phù hợp với dữ liệu mất cân bằng (**ROC-AUC, PR-AUC**), và giải thích quyết định mô hình bằng **Permutation Importance** & **Decision Tree**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (StratifiedKFold, cross_validate, cross_val_predict,
                                    train_test_split, RandomizedSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                            classification_report, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Đã kết nối Python thành công trên VS Code!")

## 1. Đọc dữ liệu & làm sạch

In [ ]:
DATA_PATH = "../data/WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df["Attrition"].value_counts(normalize=True).round(3))

# Cột hằng số (không mang thông tin) và cột ID
drop_cols = [c for c in df.columns if df[c].nunique() == 1] + ["EmployeeNumber"]
print("Bỏ các cột:", drop_cols)
df = df.drop(columns=drop_cols)

## 2. Exploratory Data Analysis (EDA)
Phân tích ban đầu các biến chính ảnh hưởng đến quyết định nghỉ việc.

### 2.1. Phân bố tỷ lệ hài lòng công việc và hài lòng môi trường

In [ ]:
# Tỷ lệ nghỉ việc theo JobSatisfaction (giữ thứ tự tự nhiên 1→4)
job_sat = (
    df[["JobSatisfaction"]]
    .assign(Attrition=(df["Attrition"] == "Yes").astype(int))
    .groupby("JobSatisfaction", sort=False)
    .agg(
        employee_count=("Attrition", "size"),
        attrition_rate=("Attrition", "mean")
    )
    .reset_index()
).sort_values("JobSatisfaction")

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=job_sat, x="JobSatisfaction", y="attrition_rate", color="steelblue", order=job_sat["JobSatisfaction"])
for i, (_, row) in enumerate(job_sat.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}",
        (i, row['attrition_rate']),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
ax.set(title="Tỷ lệ nghỉ việc theo Mức độ hài lòng công việc (JobSatisfaction)",
       xlabel="Mức độ hài lòng (1=Rất thấp, 4=Rất cao)",
       ylabel="Tỷ lệ nghỉ việc")
ax.set_ylim(0, job_sat["attrition_rate"].max() * 1.15)
plt.tight_layout()
plt.show()

Nhân viên có mức độ hài lòng công việc ở mức 1 (Rất thấp) có tỷ lệ nghỉ việc rất cao (~40%), trong khi những người hài lòng mức 4 (Rất cao) chỉ có khoảng 11%. Sự khác biệt này cho thấy hài lòng công việc là yếu tố quan trọng ảnh hưởng trực tiếp đến quyết định ở lại hay rời đi.

In [ ]:
# Tỷ lệ nghỉ việc theo EnvironmentSatisfaction (giữ thứ tự tự nhiên 1→4)
env_sat = (
    df[["EnvironmentSatisfaction"]]
    .assign(Attrition=(df["Attrition"] == "Yes").astype(int))
    .groupby("EnvironmentSatisfaction", sort=False)
    .agg(
        employee_count=("Attrition", "size"),
        attrition_rate=("Attrition", "mean")
    )
    .reset_index()
).sort_values("EnvironmentSatisfaction")

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=env_sat, x="EnvironmentSatisfaction", y="attrition_rate", color="coral", order=env_sat["EnvironmentSatisfaction"])
for i, (_, row) in enumerate(env_sat.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}",
        (i, row['attrition_rate']),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
ax.set(title="Tỷ lệ nghỉ việc theo Mức độ hài lòng môi trường (EnvironmentSatisfaction)",
       xlabel="Mức độ hài lòng (1=Rất thấp, 4=Rất cao)",
       ylabel="Tỷ lệ nghỉ việc")
ax.set_ylim(0, env_sat["attrition_rate"].max() * 1.15)
plt.tight_layout()
plt.show()

Tương tự như hài lòng công việc, nhân viên không hài lòng với môi trường làm việc (mức 1) có xu hướng nghỉ việc cao hơn (~37%) so với những người hài lòng mức 4 (~8%). Điều này nhấn mạnh rằng điều kiện làm việc là yếu tố then chốt giữ chân nhân viên.

### 2.2. Ảnh hưởng của làm thêm giờ (OverTime) và thu nhập

In [ ]:
# Tỷ lệ nghỉ việc theo OverTime
overtime_data = (
    df[["OverTime"]]
    .assign(Attrition=(df["Attrition"] == "Yes").astype(int))
    .groupby("OverTime")
    .agg(
        employee_count=("Attrition", "size"),
        attrition_rate=("Attrition", "mean")
    )
    .reindex(["Yes", "No"])
    .reset_index()
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=overtime_data, x="OverTime", y="attrition_rate", color="crimson")
for bar, (_, row) in zip(ax.patches, overtime_data.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
ax.set(title="Tỷ lệ nghỉ việc theo tình trạng làm thêm giờ",
       xlabel="Làm thêm giờ (OverTime)",
       ylabel="Tỷ lệ nghỉ việc")
ax.set_ylim(0, overtime_data["attrition_rate"].max() * 1.15)
plt.tight_layout()
plt.show()

Nhân viên làm thêm giờ có tỷ lệ nghỉ việc cao gấp 3 lần (34.2%) so với nhóm không làm thêm giờ (11.5%). Điều này cho thấy áp lực công việc quá tải (OverTime) là một nguyên nhân chính khiến nhân viên rời bỏ công ty.

### 2.3. Phân bố lương hàng tháng (MonthlyIncome)

In [ ]:
# So sánh phân bố lương giữa nhóm nghỉ việc và không
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

attrition_status = ["No", "Yes"]
for ax, status in zip(axes, attrition_status):
    income_data = df[df["Attrition"] == status]["MonthlyIncome"]
    ax.hist(income_data, bins=30, edgecolor="black", alpha=0.7, color="skyblue")
    ax.set_xlabel("Thu nhập hàng tháng ($)")
    ax.set_ylabel("Số nhân viên")
    ax.set_title(f"Phân bố lương - Attrition={status}\n(Trung bình: ${income_data.mean():.0f})")
    ax.axvline(income_data.mean(), color="red", linestyle="--", linewidth=2, label="Trung bình")
    ax.legend()

plt.tight_layout()
plt.show()

Nhóm nhân viên nghỉ việc có phân bố lương tập trung ở mức thấp (trung bình ~$2,271/tháng) trong khi nhóm ở lại có lương trung bình cao hơn (~$6,502/tháng). Sự chênh lệch lương lớn này là yếu tố quan trọng khiến nhân viên lương thấp dễ rời bỏ công ty hơn.

### 2.4. Ảnh hưởng của khoảng cách từ nhà đến công ty (DistanceFromHome)

In [ ]:
# Phân nhóm khoảng cách
df_temp = df.copy()
df_temp["Attrition"] = (df["Attrition"] == "Yes").astype(int)
df_temp["DistanceGroup"] = pd.cut(df_temp["DistanceFromHome"], 
                                    bins=[0, 5, 10, 30], 
                                    labels=["0-5 km", "5-10 km", ">10 km"])

distance_attrition = df_temp.groupby("DistanceGroup").agg(
    employee_count=("Attrition", "size"),
    attrition_rate=("Attrition", "mean")
).reset_index()

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=distance_attrition, x="DistanceGroup", y="attrition_rate", color="mediumseagreen")
for bar, (_, row) in zip(ax.patches, distance_attrition.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
ax.set(title="Tỷ lệ nghỉ việc theo khoảng cách từ nhà (DistanceFromHome)",
       xlabel="Khoảng cách",
       ylabel="Tỷ lệ nghỉ việc")
ax.set_ylim(0, distance_attrition["attrition_rate"].max() * 1.15)
plt.tight_layout()
plt.show()

Tỷ lệ nghỉ việc tăng theo khoảng cách: nhân viên sống xa công ty (>10 km) có tỷ lệ nghỉ việc cao gấp 2 lần so với nhân viên sống gần (0-5 km). Khoảng cách đi làm dài kéo theo chi phí thời gian và năng lượng cao, ảnh hưởng tiêu cực đến quyết định ở lại.

## 3. Feature Engineering
Các biến phái sinh phản ánh mức lương so với cấp bậc, mức độ gắn bó và tình trạng thăng tiến.

In [ ]:
df["IncomePerLevel"]     = df["MonthlyIncome"] / df["JobLevel"]
df["LoyaltyRatio"]       = df["YearsAtCompany"] / (df["TotalWorkingYears"] + 1)
df["PromoGapRatio"]      = df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)
df["AvgSatisfaction"]    = df[["JobSatisfaction", "EnvironmentSatisfaction",
                               "RelationshipSatisfaction", "WorkLifeBalance"]].mean(axis=1)
df["OT_LowSatisfaction"] = ((df["OverTime"] == "Yes") & (df["JobSatisfaction"] <= 2)).astype(int)
df["JobHopper"]          = df["NumCompaniesWorked"] / (df["TotalWorkingYears"] + 1)

y = (df.pop("Attrition") == "Yes").astype(int)
X = df

# BusinessTravel có thứ tự tự nhiên -> ordinal; các biến chữ còn lại -> one-hot
ord_cols = ["BusinessTravel"]
travel_order = [["Non-Travel", "Travel_Rarely", "Travel_Frequently"]]
cat_cols = [c for c in X.select_dtypes("object").columns if c not in ord_cols]
num_cols = [c for c in X.columns if c not in cat_cols + ord_cols]

def make_preprocessor(scale=True):
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num_cols),
        ("ord", OrdinalEncoder(categories=travel_order), ord_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])

## 4. Chia train/test & so sánh mô hình bằng Stratified CV
**Lưu ý:** với ~84% nhãn `No`, mô hình đoán "No" cho tất cả đã đạt ~84% accuracy, nên ta so sánh bằng ROC-AUC, PR-AUC, Recall và F1 của lớp `Yes`.
Test set được giữ riêng, chỉ dùng một lần ở cuối.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "LogReg": Pipeline([("pre", make_preprocessor(True)),
                        ("m", LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000))]),
    "RandomForest": Pipeline([("pre", make_preprocessor(False)),
                              ("m", RandomForestClassifier(n_estimators=500, max_depth=8,
                                    min_samples_leaf=5, class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1))]),
    "XGBoost": Pipeline([("pre", make_preprocessor(False)),
                         ("m", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
                                eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["roc_auc", "average_precision", "recall", "f1"]

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({"model": name, **{s: res[f"test_{s}"].mean() for s in scoring},
                 "roc_auc_std": res["test_roc_auc"].std()})
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

## 5. Tinh chỉnh siêu tham số (RandomizedSearchCV, tối ưu ROC-AUC)

In [ ]:
param_dists = {
    "LogReg": {"m__C": np.logspace(-3, 1, 20)},
    "RandomForest": {"m__n_estimators": [300, 500, 800],
                     "m__max_depth": [4, 6, 8, 12, None],
                     "m__min_samples_leaf": [2, 5, 10, 20],
                     "m__max_features": ["sqrt", 0.3, 0.5]},
    "XGBoost": {"m__n_estimators": [150, 300, 500],
                "m__max_depth": [2, 3, 4, 5],
                "m__learning_rate": [0.01, 0.03, 0.05, 0.1],
                "m__subsample": [0.6, 0.8, 1.0],
                "m__colsample_bytree": [0.6, 0.8, 1.0],
                "m__min_child_weight": [1, 3, 5],
                "m__reg_lambda": [1, 5, 10]},
}

best = {}
for name, pipe in models.items():
    search = RandomizedSearchCV(pipe, param_dists[name], n_iter=15, scoring="roc_auc",
                                cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    best[name] = search
    print(f"{name:13s} CV ROC-AUC = {search.best_score_:.3f}  | {search.best_params_}")

## 6. Chọn ngưỡng quyết định
Ngưỡng 0.5 mặc định hiếm khi phù hợp với dữ liệu mất cân bằng. Ta lấy xác suất out-of-fold trên tập train rồi chọn ngưỡng theo mục tiêu của HR (ưu tiên phát hiện nhân viên nghỉ việc, target recall ≥ 70%).

In [ ]:
best_name = max(best, key=lambda k: best[k].best_score_)
best_model = best[best_name].best_estimator_
print("Mô hình tốt nhất:", best_name)

oof_proba = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_proba)

TARGET_RECALL = 0.70
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
idx = ok[np.argmax(prec[:-1][ok])] if len(ok) else np.argmax(2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9))
threshold = thr[idx]
print(f"Ngưỡng chọn = {threshold:.3f} | OOF precision = {prec[idx]:.3f}, recall = {rec[idx]:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(rec, prec); plt.scatter(rec[idx], prec[idx], c="red", label=f"thr={threshold:.2f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–Recall (out-of-fold)")
plt.legend(); plt.show()

## 7. Đánh giá cuối cùng trên tập test (chạm test set một lần)

In [ ]:
proba_test = best_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= threshold).astype(int)

print(f"ROC-AUC : {roc_auc_score(y_test, proba_test):.3f}")
print(f"PR-AUC  : {average_precision_score(y_test, proba_test):.3f}  (baseline = {y_test.mean():.3f})")
print(classification_report(y_test, pred_test, target_names=["No", "Yes"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=["No", "Yes"]).plot(cmap="Blues")
plt.show()

## 8. Độ quan trọng đặc trưng (Permutation Importance)
Impurity importance của Random Forest thiên lệch về biến số liên tục có nhiều giá trị. Permutation importance trên tập test đáng tin cậy hơn. Thanh sai số được vẽ dựa trên cột 'std' từ `permutation_importance`.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring="roc_auc",
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"feature": X_test.columns, "mean": perm.importances_mean, "std": perm.importances_std}).sort_values("mean", ascending=False)

print("\n=== BẢNG PERMUTATION IMPORTANCE ĐẦY ĐỦ (Tập test) ===")
print(imp.to_string(index=False))

print("\n=== TOP 5 ĐẶC TRƯNG ===")
for idx, row in imp.head(5).iterrows():
    print(f"{row['feature']:25s}: {row['mean']:8.4f} ± {row['std']:7.4f}")

# Vẽ biểu đồ Top 15 có thanh sai số
imp_top15 = imp.head(15)
plt.figure(figsize=(8, 6))
ax = plt.barh(range(len(imp_top15)), imp_top15["mean"].values, 
               xerr=imp_top15["std"].values, color="steelblue", capsize=5)
ax.set_yticks(range(len(imp_top15)))
ax.set_yticklabels(imp_top15["feature"].values)
ax.set_xlabel("Giảm ROC-AUC khi xáo trộn đặc trưng")
ax.set_title("Top 15 đặc trưng quan trọng nhất (Permutation Importance với sai số)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Feature Importance - Mô hình Decision Tree (max_depth=5)
Decision Tree với chiều sâu giới hạn (max_depth=5) cung cấp mô hình đơn giản và dễ diễn giải. Huấn luyện trên toàn bộ 1.470 bản ghi.

In [ ]:
# Huấn luyện Decision Tree với max_depth=5 trên toàn bộ dataset
dt_pipeline = Pipeline([
    ("pre", make_preprocessor(False)),
    ("m", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE))
])
dt_pipeline.fit(X, y)

# Lấy feature importance từ Decision Tree
preprocessor = dt_pipeline.named_steps["pre"]
dt_model = dt_pipeline.named_steps["m"]

# Lấy tên các đặc trưng sau khi transform
feature_names = []
num_features = preprocessor.named_transformers_["num"].get_feature_names_out(num_cols).tolist()
ord_features = preprocessor.named_transformers_["ord"].get_feature_names_out(ord_cols).tolist()
cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()

feature_names = num_features + ord_features + cat_features

# Tính toán feature importance (normalized)
importances = dt_model.feature_importances_
dt_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
    "importance_pct": (importances * 100).round(4)
}).sort_values("importance", ascending=False)

print("\n=== TOP 10 FEATURE IMPORTANCE (Decision Tree, max_depth=5, toàn bộ 1.470 bản ghi) ===")
print(dt_imp.head(10).to_string(index=False))

# Vẽ biểu đồ
plt.figure(figsize=(8, 6))
sns.barplot(data=dt_imp.head(10), x="importance_pct", y="feature", color="coral")
plt.xlabel("Độ quan trọng (%)"); plt.title("Top 10 Feature Importance - Decision Tree (max_depth=5)")
plt.tight_layout(); plt.show()

# Chi tiết cho top 5
print("\n=== CHI TIẾT TOP 5 ĐẶC TRƯNG ===")
for idx, row in dt_imp.head(5).iterrows():
    print(f"{row['feature']:20s}: {row['importance_pct']:7.4f}%")

## 10. Phương pháp so sánh tỷ lệ nghỉ việc theo phòng ban
Tỷ lệ nghỉ việc của mỗi phòng ban được tính bằng số nhân viên nghỉ việc chia cho tổng số nhân viên trong phòng ban đó. So sánh các tỷ lệ này giúp xác định phòng ban nào cần sự chú ý khẩn cấp.

In [ ]:
# Tỷ lệ nghỉ việc theo phòng ban
dept_attrition = (
    X[["Department"]]
    .assign(Attrition=y)
    .groupby("Department")
    .agg(
        attrition_rate=("Attrition", "mean"),
        employee_count=("Attrition", "size")
    )
    .sort_values("attrition_rate", ascending=False)
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=dept_attrition.reset_index(),
    x="Department",
    y="attrition_rate",
    color="steelblue"
)

# Hiển thị tỷ lệ và số nhân viên trên từng cột
for bar, (_, row) in zip(ax.patches, dept_attrition.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}\n(n={row['employee_count']})",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=10,
        xytext=(0, 4), textcoords="offset points"
    )

ax.set(
    title="Tỷ lệ nghỉ việc theo phòng ban",
    xlabel="Phòng ban",
    ylabel="Tỷ lệ nghỉ việc"
)
ax.set_ylim(0, dept_attrition["attrition_rate"].max() * 1.2)
plt.tight_layout()
plt.show()

## 11. So sánh tỷ lệ nghỉ việc theo tình trạng làm ngoài giờ (OverTime)
Tỷ lệ nghỉ việc của mỗi nhóm được tính bằng số nhân viên nghỉ việc chia cho tổng số nhân viên trong nhóm `OverTime` tương ứng (`Yes` hoặc `No`). So sánh để hiểu ảnh hưởng của làm ngoài giờ.

In [ ]:
# Tính tỷ lệ nghỉ việc theo tình trạng làm ngoài giờ (OverTime)
attrition_by_overtime = (
    X[["OverTime"]]
    .assign(Attrition=y)
    .groupby("OverTime")
    .agg(
        employee_count=("Attrition", "size"),
        attrition_count=("Attrition", "sum"),
        attrition_rate=("Attrition", "mean")
    )
    .reindex(["Yes", "No"])
    .reset_index()
)

print("=== TỶ LỆ NGHỈ VIỆC THEO TÌNH TRẠNG LÀM NGOÀI GIỜ ===")
print(attrition_by_overtime.to_string(index=False))

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=attrition_by_overtime,
    x="OverTime",
    y="attrition_rate",
    color="darkorange"
)

for bar, (_, row) in zip(ax.patches, attrition_by_overtime.iterrows()):
    ax.annotate(
        f"{row['attrition_rate']:.1%}\n(n={row['employee_count']})",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=10,
        xytext=(0, 4), textcoords="offset points"
    )

ax.set(
    title="Tỷ lệ nghỉ việc theo tình trạng làm ngoài giờ",
    xlabel="Làm ngoài giờ (OverTime)",
    ylabel="Tỷ lệ nghỉ việc"
)
ax.set_ylim(0, attrition_by_overtime["attrition_rate"].max() * 1.2)
plt.tight_layout()
plt.show()